In [1]:
import sys, json, warnings
sys.path.insert(0, "..")            # ml-service root, so `training` and `app` import
warnings.filterwarnings("ignore")
import duckdb, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from training import config as C
sns.set_theme(style="whitegrid")
IMG = C.ROOT_DIR / "docs" / "images"; IMG.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(); con.execute("SET memory_limit='1500MB'"); con.execute("SET threads=2")
G = str(C.FLIGHTS_GLOB).replace("\\", "/")
def q(sql): return con.execute(sql.replace("{G}", G)).df()
def save(name):
    plt.tight_layout(); plt.savefig(IMG / f"{name}.png", dpi=110); plt.show()
from training.feature_engineering import load_xy
from app.services.feature_builder import FEATURE_COLUMNS

# 02 - Cleaning and features
Cleaning is done in DuckDB by `training/data_cleaning.py` (see its docstring). Here we verify the result and inspect features.

In [2]:
print(len(FEATURE_COLUMNS), "features"); print(FEATURE_COLUMNS)
for sp in ("train", "valid", "test"):
    X, y = load_xy(sp, "delay", sample=0.05)
    print(f"{sp:6s} rows(5%)={len(X):7d} nan_cells={int(X.isna().sum().sum())} delay_rate={y.mean():.3f}")

33 features
['dep_time_idx', 'day_of_week', 'is_weekend', 'is_holiday', 'holiday_distance', 'distance_miles', 'airline_freq', 'origin_freq', 'dest_freq', 'route_freq', 'route_daily_avg', 'origin_daily_avg', 'airline_delay_roll', 'origin_delay_roll', 'dest_delay_roll', 'airline_cancel_roll', 'origin_cancel_roll', 'dest_cancel_roll', 'route_delay_rate', 'origin_time_delay_rate', 'o_prcp_band', 'o_snow_band', 'o_wind_band', 'o_freeze', 'o_severity', 'o_tavg', 'd_prcp_band', 'd_snow_band', 'd_wind_band', 'd_freeze', 'd_severity', 'd_tavg', 'weather_missing']


train  rows(5%)= 222856 nan_cells=0 delay_rate=0.230
valid  rows(5%)=  28044 nan_cells=0 delay_rate=0.184


test   rows(5%)=  86267 nan_cells=0 delay_rate=0.149


## Leakage check
Aircraft columns are NULL for every cancelled flight (they come only from the main file), so they would reveal the cancel label and are NOT features. Same for all post-departure columns.

In [3]:
q("select is_cancelled, count(*) n, count(manufacturer) with_aircraft_info from read_parquet('{G}') group by 1")

,is_cancelled,n,with_aircraft_info
0,0,6743361,6743361
1,1,86991,0


In [4]:
X, y = load_xy("train", "delay", sample=0.05)
corr = pd.Series({c: np.corrcoef(X[c], y)[0, 1] for c in X.columns}).sort_values(key=abs, ascending=False)
fig, ax = plt.subplots(figsize=(7, 8)); corr.plot.barh(ax=ax, color="#3b82f6"); ax.invert_yaxis(); ax.set_title("correlation of each feature with is_delayed (train, 5%)")
save("feature_correlation")

In [5]:
Xt, yt = load_xy("test", "delay", sample=0.05)
shift = pd.DataFrame({"train_mean": X[["airline_delay_roll","origin_delay_roll","o_severity","d_severity"]].mean(),
                      "test_mean": Xt[["airline_delay_roll","origin_delay_roll","o_severity","d_severity"]].mean()})
print("test uses the shipped 2023-10-01 rolling snapshot (what a deployed model without daily refresh sees)"); shift

test uses the shipped 2023-10-01 rolling snapshot (what a deployed model without daily refresh sees)


,train_mean,test_mean
airline_delay_roll,0.233983,0.188044
origin_delay_roll,0.234416,0.189485
o_severity,0.765086,0.604310
d_severity,0.761458,0.603232
